In [8]:
import mlflow
import nltk
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('stopwords')
import numpy as np

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Abcom\AppData\Roaming\nltk_data...
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Abcom\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.head()

,review,sentiment
791,This is a great British film. A cleverly obser...,positive
609,1st watched 12/24/2009  4 out of 10 (Dir-Robe...,negative
259,I am a massive fan of the book and Orwell is c...,negative
892,"On many levels it's very good. In fact, consid...",positive
601,I suppose if you like pure action... you'll fi...,negative


In [6]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\Abcom\AppData\Local\Temp\ipykernel_3168\3798096103.py:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  text = re.sub('\s+', ' ', text).strip()


In [9]:
df = normalize_text(df)
df.head()

,review,sentiment
791,great british film cleverly observed script ma...,positive
609,st watched  dir robert elli miller emotional ...,negative
259,massive fan book orwell certainly favourite wr...,negative
892,many level good fact considering low budget br...,positive
601,suppose like pure action find suppose find gor...,negative


In [10]:
df['sentiment'].value_counts()

sentiment
negative    267
positive    233
Name: count, dtype: int64

In [11]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [12]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
791,great british film cleverly observed script ma...,1
609,st watched  dir robert elli miller emotional ...,0
259,massive fan book orwell certainly favourite wr...,0
892,many level good fact considering low budget br...,1
601,suppose like pure action find suppose find gor...,0


In [13]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [14]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [19]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/gurpreet-singh-ji/mlops-adv-proj.mlflow')
dagshub.init(repo_owner='gurpreet-singh-ji', repo_name='mlops-adv-proj', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6bb72ba4-f0ab-4d25-ba25-d6720a4ea24f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7852164c2f4e78a789f668d810a814aeb5ab5cc5f10b8b86f99242dd61fa6914




Accessing as gurpreet-singh-ji

Initialized MLflow to track repo "gurpreet-singh-ji/mlops-adv-proj"

Repository gurpreet-singh-ji/mlops-adv-proj initialized!

2026/05/09 12:20:35 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/89932c71c91b48ce9e0ba2dc790b0f0d', creation_time=1778309429130, experiment_id='0', last_update_time=1778309429130, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [20]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-05-09 12:20:46,715 - INFO - Starting MLflow run...
2026-05-09 12:20:47,846 - INFO - Logging preprocessing parameters...
2026-05-09 12:20:49,135 - INFO - Initializing Logistic Regression model...
2026-05-09 12:20:49,138 - INFO - Fitting the model...
2026-05-09 12:20:49,285 - INFO - Model training complete.
2026-05-09 12:20:49,289 - INFO - Logging model parameters...
2026-05-09 12:20:49,605 - INFO - Making predictions...
2026-05-09 12:20:49,608 - INFO - Calculating evaluation metrics...
2026-05-09 12:20:49,672 - INFO - Logging evaluation metrics...
2026-05-09 12:20:50,980 - INFO - Saving and logging the model...
2026/05/09 12:20:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 12:20:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The rec

🏃 View run painted-yak-186 at: https://dagshub.com/gurpreet-singh-ji/mlops-adv-proj.mlflow/#/experiments/0/runs/3d7fc5d3db154d348882d80585bdc1f1
🧪 View experiment at: https://dagshub.com/gurpreet-singh-ji/mlops-adv-proj.mlflow/#/experiments/0
